In [1]:
# ============================================================
# Simulation Experiment (Regression) — Filtered Setting Only
# 20 Repetitions with mean ± SE reporting
# ============================================================

import os
import sys
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.impute import KNNImputer

sys.path.append('../')
from meta_fusion.benchmarks import Benchmarks  # kept for imputation blocks
from meta_fusion.methods import Trainer, EnsembleSelection
from meta_fusion.models import MLP_Net
from meta_fusion.utils import (
    AverageMeter, get_weights_by_task_loss, set_random_seed, CustomDataset
)
from meta_fusion.synthetic_data import PrepareSyntheticData
from meta_fusion.config import load_config
from meta_fusion.methodsextra import Extractors, Cohorts  # kept for imputation blocks
from meta_fusion.methodsextra_new import (
    Trainer_Joint_new, Trainer_new, Cohorts_new, EnsembleSelection_new
)

# ============================================================
# EXPERIMENT PARAMETERS
# ============================================================

SEED             = 1234
NUM_REPETITIONS  = 20
REPETITION_SEEDS = [SEED + i for i in range(NUM_REPETITIONS)]

N                = 2000
DIM_MODALITIES   = [200, 300, 100]
DIM_LATENT       = [20, 30, 10, 0]           # last = shared component
NOISE_RATIOS     = [0.6, 0.1, 0.1]
TRANS_TYPE       = ["linear", "quadratic", "quadratic", "linear"]
MOD_PROP         = [1, 1, 1, 0, 0]
INTERACTIVE_PROP = 0
FRACTIONS        = [1.0, 0.8, 0.6]
MISSING_VALUE    = 10.0

NUM_MODALITIES   = len(DIM_MODALITIES)
COMBINED_HIDDENS = [128, 64]                  # used in BenchmarksLateFusion MLPs
MOD_HIDDENS      = [[256], [256], [256]]      # used in Cohorts_new

OUTPUT_DIM = 1
DATA_NAME  = "regression"

USE_GPU = torch.cuda.is_available()

KNN_NEIGHBORS = 5
KNN_WEIGHTS   = "distance"

OUTDIR    = "./results/simulation_full/"
CKPT_ROOT = "./checkpoints/simulation_full/"
os.makedirs(OUTDIR,    exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)

CONFIG_PATH           = './experiments_synthetic/config.json'
EXTRACTOR_CONFIG_PATH = './experiments_synthetic/config_extractor.json'


# ============================================================
# CONFIG FACTORY
# ============================================================

def make_config(split_seed: int):
    config           = load_config(CONFIG_PATH)
    extractor_config = load_config(EXTRACTOR_CONFIG_PATH)

    ckpt_dir = os.path.join(CKPT_ROOT, f"split_seed_{split_seed}")
    os.makedirs(ckpt_dir, exist_ok=True)

    for cfg in (config, extractor_config):
        cfg['ckpt_dir']     = ckpt_dir
        cfg['output_dim']   = OUTPUT_DIM
        cfg['random_state'] = split_seed

    config['use_gpu']   = USE_GPU
    config['init_lr']   = 0.001
    # Regression-only methods. Trainer_Joint_new.test_regression rejects
    # majority_voting / weighted_voting, so we don't put them here. The
    # benchmarks side (BenchmarksLateFusion) implements its own ensemble
    # sweep that does include median-based "voting" analogs.
    config['ensemble_methods'] = [
        "simple_average",
        "weighted_average",
        "greedy_ensemble",
    ]

    extractor_config['init_lr']      = [0.001] * NUM_MODALITIES
    extractor_config['weight_decay'] = [0]     * NUM_MODALITIES

    return config, extractor_config


# ============================================================
# REPRODUCIBILITY
# ============================================================

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if torch.backends.cudnn.is_available():
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark     = False


# ============================================================
# CUSTOM BENCHMARKS WITH FULL LATE-FUSION ENSEMBLE SWEEP
# ============================================================

class BenchmarksLateFusion:
    """
    Self-contained benchmark trainer for the regression simulation.

    Trains:
      - one unimodal MLP per modality (input = single modality tensor)
      - one early-fusion MLP (input = concatenated modalities)

    Each MLP is trained for `epochs` epochs with early-stopping on
    validation MSE (the model state that achieved the best val MSE
    is restored at the end of training).

    test() returns a dict of method_name -> MSE with rows:
      modality_1, modality_2, modality_3
      early_fusion
      late_fusion_simple_average
      late_fusion_weighted_average
      late_fusion_best_single
      late_fusion_greedy_ensemble
      late_fusion_majority_voting   (median of unimodal predictions)
      late_fusion_weighted_voting   (weighted median)
    """

    def __init__(self, config):
        self.use_gpu = bool(config['use_gpu'])
        self.device  = torch.device(
            'cuda' if self.use_gpu and torch.cuda.is_available() else 'cpu'
        )
        self.epochs    = int(config['epochs'])
        self.lr        = float(config['init_lr'])
        self.wd        = float(config['weight_decay'])
        self.criterion = nn.MSELoss()
        self.verbose   = bool(config.get('verbose', True))

        # one MLP per modality
        self.unimodal_models = [
            MLP_Net(int(DIM_MODALITIES[i]), COMBINED_HIDDENS, OUTPUT_DIM).to(self.device)
            for i in range(NUM_MODALITIES)
        ]
        # early-fusion MLP over concatenated modalities
        total_dim = int(sum(DIM_MODALITIES))
        self.early_fusion_model = MLP_Net(
            total_dim, COMBINED_HIDDENS, OUTPUT_DIM
        ).to(self.device)

        # optimizers
        self.unimodal_optimizers = [
            optim.Adam(m.parameters(), lr=self.lr, weight_decay=self.wd)
            for m in self.unimodal_models
        ]
        self.early_fusion_optimizer = optim.Adam(
            self.early_fusion_model.parameters(),
            lr=self.lr, weight_decay=self.wd,
        )

        # state populated by train()
        self.unimodal_val_losses    = [float('inf')] * NUM_MODALITIES
        self.early_fusion_val_loss  = float('inf')
        self.greedy_subset          = list(range(NUM_MODALITIES))

    # ---- training utilities ----

    def _to_device(self, mods, target):
        if self.use_gpu:
            mods   = [m.cuda() for m in mods]
            target = target.cuda()
        return mods, target

    @staticmethod
    def _ensure_2d(y):
        return y.unsqueeze(-1) if y.dim() == 1 else y

    def _train_one_model(self, model, optimizer, train_loader, val_loader, get_input):
        """
        Trains `model` for self.epochs epochs with best-val checkpointing.
        get_input(mods) -> tensor: builds the model input from a list of modality tensors.
        Returns the best validation MSE.
        """
        best_state = copy.deepcopy(model.state_dict())
        best_val   = float('inf')

        for ep in range(self.epochs):
            # train
            model.train()
            for batch in train_loader:
                mods, target = batch[:-1], batch[-1]
                mods, target = self._to_device(mods, target)

                x = get_input(mods)
                y = self._ensure_2d(target.float())

                pred = model(x)
                loss = self.criterion(pred, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            # validate
            model.eval()
            total, n = 0.0, 0
            with torch.no_grad():
                for batch in val_loader:
                    mods, target = batch[:-1], batch[-1]
                    mods, target = self._to_device(mods, target)

                    x = get_input(mods)
                    y = self._ensure_2d(target.float())

                    pred = model(x)
                    total += float(self.criterion(pred, y).item()) * y.size(0)
                    n     += y.size(0)
            val_mse = total / n if n > 0 else float('inf')

            if val_mse < best_val:
                best_val   = val_mse
                best_state = copy.deepcopy(model.state_dict())

        model.load_state_dict(best_state)
        return best_val

    # ---- prediction utilities ----

    def _unimodal_predictions(self, loader):
        """Returns (NUM_MODALITIES, N) preds and (N,) targets, both float32 numpy."""
        for m in self.unimodal_models:
            m.eval()
        per_mod = [[] for _ in range(NUM_MODALITIES)]
        targets = []
        with torch.no_grad():
            for batch in loader:
                mods, target = batch[:-1], batch[-1]
                mods, target = self._to_device(mods, target)
                for i in range(NUM_MODALITIES):
                    p = self.unimodal_models[i](mods[i])
                    per_mod[i].append(p.detach().cpu().numpy().reshape(-1))
                targets.append(target.detach().cpu().numpy().reshape(-1))
        preds   = np.stack([np.concatenate(p) for p in per_mod], axis=0).astype(np.float32)
        targets = np.concatenate(targets).astype(np.float32)
        return preds, targets

    def _early_fusion_predictions(self, loader):
        """Returns (N,) preds and (N,) targets."""
        self.early_fusion_model.eval()
        preds, targets = [], []
        with torch.no_grad():
            for batch in loader:
                mods, target = batch[:-1], batch[-1]
                mods, target = self._to_device(mods, target)
                x = torch.cat([m for m in mods[:NUM_MODALITIES]], dim=1)
                p = self.early_fusion_model(x)
                preds.append(p.detach().cpu().numpy().reshape(-1))
                targets.append(target.detach().cpu().numpy().reshape(-1))
        return (
            np.concatenate(preds).astype(np.float32),
            np.concatenate(targets).astype(np.float32),
        )

    # ---- greedy ensemble selection ----

    def _greedy_select(self, val_loader):
        """
        Greedy forward selection on val MSE.
        Start with the single model with the lowest val MSE; iteratively
        add the model whose inclusion (via simple averaging) most reduces
        val MSE, stopping when no addition helps.
        """
        preds, targets = self._unimodal_predictions(val_loader)  # (M, N), (N,)

        best_first = int(np.argmin(self.unimodal_val_losses))
        chosen     = [best_first]
        chosen_set = {best_first}
        cur_pred   = preds[best_first].copy()
        cur_mse    = float(np.mean((cur_pred - targets) ** 2))

        improved = True
        while improved:
            improved   = False
            best_new   = None
            best_mse   = cur_mse
            for i in range(NUM_MODALITIES):
                if i in chosen_set:
                    continue
                k       = len(chosen)
                trial   = (cur_pred * k + preds[i]) / (k + 1)
                trial_m = float(np.mean((trial - targets) ** 2))
                if trial_m < best_mse:
                    best_mse = trial_m
                    best_new = i
            if best_new is not None:
                k          = len(chosen)
                cur_pred   = (cur_pred * k + preds[best_new]) / (k + 1)
                cur_mse    = best_mse
                chosen.append(best_new)
                chosen_set.add(best_new)
                improved   = True
        return chosen

    # ---- weighted median ----

    @staticmethod
    def _weighted_median(preds, weights):
        """
        preds:   (M, N) float
        weights: (M,) float, non-negative, will be normalized to sum 1
        Returns: (N,) weighted median across the M models per sample.
        """
        w = np.asarray(weights, dtype=np.float64)
        s = w.sum()
        if s <= 0:
            # fall back to plain median
            return np.median(preds, axis=0).astype(np.float32)
        w = w / s

        order        = np.argsort(preds, axis=0)                         # (M, N)
        sorted_preds = np.take_along_axis(preds, order, axis=0)          # (M, N)
        # broadcast w along axis 0 in sorted order
        w_col        = w.reshape(-1, 1)
        sorted_w     = np.take_along_axis(
            np.broadcast_to(w_col, preds.shape), order, axis=0
        )
        cum          = np.cumsum(sorted_w, axis=0)                       # (M, N)
        # first index where cum >= 0.5
        idx          = np.argmax(cum >= 0.5, axis=0)                     # (N,)
        N            = preds.shape[1]
        return sorted_preds[idx, np.arange(N)].astype(np.float32)

    # ---- public API ----

    def train(self, train_loader, val_loader):
        if self.verbose:
            print("=" * 60)
            print("Training BenchmarksLateFusion (regression)")
            print("=" * 60)

        # unimodal
        for i in range(NUM_MODALITIES):
            if self.verbose:
                print(f"  Training unimodal model {i + 1}/{NUM_MODALITIES} "
                      f"(input dim = {DIM_MODALITIES[i]})")
            self.unimodal_val_losses[i] = self._train_one_model(
                self.unimodal_models[i],
                self.unimodal_optimizers[i],
                train_loader, val_loader,
                get_input=lambda mods, idx=i: mods[idx],
            )
            if self.verbose:
                print(f"    best val MSE = {self.unimodal_val_losses[i]:.4f}")

        # early fusion
        if self.verbose:
            print(f"  Training early-fusion model "
                  f"(input dim = {sum(DIM_MODALITIES)})")
        self.early_fusion_val_loss = self._train_one_model(
            self.early_fusion_model,
            self.early_fusion_optimizer,
            train_loader, val_loader,
            get_input=lambda mods: torch.cat(
                [m for m in mods[:NUM_MODALITIES]], dim=1
            ),
        )
        if self.verbose:
            print(f"    best val MSE = {self.early_fusion_val_loss:.4f}")

        # greedy subset on val
        self.greedy_subset = self._greedy_select(val_loader)
        if self.verbose:
            print(f"  Greedy ensemble subset (val-MSE selected): "
                  f"{self.greedy_subset}")

    def test(self, test_loader):
        unimodal_preds, targets = self._unimodal_predictions(test_loader)  # (M, N), (N,)
        ef_preds, _             = self._early_fusion_predictions(test_loader)

        results = {}

        # per-modality
        for i in range(NUM_MODALITIES):
            results[f'modality_{i + 1}'] = float(
                np.mean((unimodal_preds[i] - targets) ** 2)
            )

        # early fusion
        results['early_fusion'] = float(np.mean((ef_preds - targets) ** 2))

        # weights for weighted methods (1 / val_loss, normalized)
        eps = 1e-8
        w   = np.array(
            [1.0 / (l + eps) for l in self.unimodal_val_losses],
            dtype=np.float64,
        )
        w   = w / w.sum() if w.sum() > 0 else np.ones_like(w) / len(w)

        # late fusion: simple average
        sa = np.mean(unimodal_preds, axis=0)
        results['late_fusion_simple_average'] = float(
            np.mean((sa - targets) ** 2)
        )

        # late fusion: weighted average
        wa = np.sum(w[:, None] * unimodal_preds, axis=0)
        results['late_fusion_weighted_average'] = float(
            np.mean((wa - targets) ** 2)
        )

        # late fusion: best single (by val MSE)
        best_idx = int(np.argmin(self.unimodal_val_losses))
        bs       = unimodal_preds[best_idx]
        results['late_fusion_best_single'] = float(
            np.mean((bs - targets) ** 2)
        )

        # late fusion: greedy subset
        if self.greedy_subset:
            ge = np.mean(unimodal_preds[self.greedy_subset], axis=0)
        else:
            ge = sa
        results['late_fusion_greedy_ensemble'] = float(
            np.mean((ge - targets) ** 2)
        )

        # late fusion: majority voting (regression analog = median)
        mv = np.median(unimodal_preds, axis=0)
        results['late_fusion_majority_voting'] = float(
            np.mean((mv - targets) ** 2)
        )

        # late fusion: weighted voting (regression analog = weighted median)
        wv = self._weighted_median(unimodal_preds, w)
        results['late_fusion_weighted_voting'] = float(
            np.mean((wv - targets) ** 2)
        )

        if self.verbose:
            for k, v in results.items():
                print(f"  Method: ({k}), Test_MSE: {v:.4f}")

        return results


# ============================================================
# COHORT BUILDER FOR JOINT METHODS
# ============================================================

def build_cohort_new():
    """Fresh Cohorts_new for joint methods."""
    return Cohorts_new(
        dim_modalities=DIM_MODALITIES,
        num_modalities=NUM_MODALITIES,
        mod_hiddens=MOD_HIDDENS,
        output_dim=OUTPUT_DIM,
    )


# Kept for reactivation of the imputation blocks; not used in the
# filtered-only path because BenchmarksLateFusion replaces this.
def build_benchmark_models_and_dims(train_loader, val_loader):
    """
    Original Cohorts/Extractors-based benchmark model builder.
    Only used when REAL IMPUTATION / IMPUTATION blocks are reactivated
    (those still call the framework's Benchmarks class).
    """
    bm_extractor = Extractors(
        [[d, 0] for d in DIM_MODALITIES],
        DIM_MODALITIES,
        train_loader,
        val_loader,
    )
    bm_extractor.get_dummy_extractors()
    bm_cohort = Cohorts(
        extractors=bm_extractor,
        combined_hidden_layers=COMBINED_HIDDENS,
        output_dim=OUTPUT_DIM,
    )
    bm_models = bm_cohort.get_cohort_models()
    _, bm_dims = bm_cohort.get_cohort_info()
    return bm_models, bm_dims


# ============================================================
# LOADER / ARRAY UTILITIES
# ============================================================

def loader_to_modality_arrays(loader):
    all_mods = [[] for _ in range(NUM_MODALITIES)]
    all_y    = []
    for batch in loader:
        mods, y = batch[:-1], batch[-1]
        for i in range(NUM_MODALITIES):
            all_mods[i].append(mods[i].numpy())
        all_y.append(y.numpy())
    arrays = [np.concatenate(m, axis=0).astype(np.float32) for m in all_mods]
    y      = np.concatenate(all_y, axis=0).astype(np.float32)
    return arrays, y


def arrays_to_loader(arrays, y, batch_size, shuffle, drop_last=False):
    N    = y.shape[0]
    data = np.concatenate(
        [a.reshape(N, -1) for a in arrays] + [y.reshape(N, -1)], axis=1
    )
    ds = CustomDataset(data, DIM_MODALITIES)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


def subset_loader_by_mask(loader, mask, batch_size):
    arrays, y = loader_to_modality_arrays(loader)
    return arrays_to_loader(
        [a[mask] for a in arrays], y[mask],
        batch_size=batch_size, shuffle=False,
    )


def get_fully_observed_mask(loader):
    masks = []
    for batch in loader:
        mods       = batch[:-1]
        batch_mask = None
        for m in mods[:NUM_MODALITIES]:
            present    = (m != MISSING_VALUE).view(m.size(0), -1).all(dim=1)
            batch_mask = present if batch_mask is None else (batch_mask & present)
        masks.append(batch_mask.numpy())
    return np.concatenate(masks, axis=0)


# ============================================================
# KNN IMPUTATION (kept for reactivation; unused in this revision)
# ============================================================

def _replace_sentinel_with_nan(x: np.ndarray) -> np.ndarray:
    x = x.copy().astype(np.float32)
    x[x == MISSING_VALUE] = np.nan
    return x


def _fill_nan_from_train_means(x_tr, x_va, x_te):
    col_means = np.nanmean(x_tr, axis=0)
    col_means = np.where(np.isnan(col_means), 0.0, col_means)

    def fill(arr):
        arr        = arr.copy()
        rows, cols = np.where(np.isnan(arr))
        if len(rows):
            arr[rows, cols] = col_means[cols]
        return arr

    return fill(x_tr), fill(x_va), fill(x_te)


def knn_impute_no_leakage(train_loader, val_loader, test_loader):
    tr_arrays, y_tr = loader_to_modality_arrays(train_loader)
    va_arrays, y_va = loader_to_modality_arrays(val_loader)
    te_arrays, y_te = loader_to_modality_arrays(test_loader)

    imp_tr, imp_va, imp_te = [], [], []
    for mi, (x_tr, x_va, x_te) in enumerate(
        zip(tr_arrays, va_arrays, te_arrays)
    ):
        print(f"  KNN imputation | modality {mi} | "
              f"train={x_tr.shape}, val={x_va.shape}, test={x_te.shape}")

        x_tr_nan = _replace_sentinel_with_nan(x_tr)
        x_va_nan = _replace_sentinel_with_nan(x_va)
        x_te_nan = _replace_sentinel_with_nan(x_te)

        x_tr_nan, x_va_nan, x_te_nan = _fill_nan_from_train_means(
            x_tr_nan, x_va_nan, x_te_nan
        )

        imputer = KNNImputer(n_neighbors=KNN_NEIGHBORS, weights=KNN_WEIGHTS)
        imp_tr.append(np.nan_to_num(imputer.fit_transform(x_tr_nan), nan=0.0).astype(np.float32))
        imp_va.append(np.nan_to_num(imputer.transform(x_va_nan),     nan=0.0).astype(np.float32))
        imp_te.append(np.nan_to_num(imputer.transform(x_te_nan),     nan=0.0).astype(np.float32))

    new_train = arrays_to_loader(imp_tr, y_tr, train_loader.batch_size, shuffle=True)
    new_val   = arrays_to_loader(imp_va, y_va, val_loader.batch_size,   shuffle=False)
    new_test  = arrays_to_loader(imp_te, y_te, test_loader.batch_size,  shuffle=False)
    return new_train, new_val, new_test


# ============================================================
# OPTION-2 AVAILABLE-ONLY LATE FUSION (kept for reactivation; unused)
# ============================================================

class _SingleModDataset(Dataset):
    def __init__(self, x: np.ndarray, y: np.ndarray):
        self.x = torch.tensor(x, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


class LateFusionAvailableOnly:
    """Used only in IMPUTATION block; preserved verbatim for reactivation."""

    def __init__(self, config):
        self.use_gpu   = bool(config['use_gpu'])
        self.device    = torch.device('cuda' if self.use_gpu
                                      and torch.cuda.is_available() else 'cpu')
        self.epochs    = int(config['epochs'])
        self.lr        = float(config['init_lr'])
        self.wd        = float(config['weight_decay'])
        self.criterion = nn.MSELoss()

        self.models = [
            MLP_Net(int(d), COMBINED_HIDDENS, OUTPUT_DIM).to(self.device)
            for d in DIM_MODALITIES
        ]
        self.optimizers = [
            optim.Adam(m.parameters(), lr=self.lr, weight_decay=self.wd)
            for m in self.models
        ]
        self.best_val_losses = [float('inf')] * NUM_MODALITIES
        self.ens_idxs        = list(range(NUM_MODALITIES))

    def _avail_loader(self, arrays, y, mi, batch_size):
        mask = ~np.all(arrays[mi] == MISSING_VALUE, axis=1)
        ds   = _SingleModDataset(arrays[mi][mask], y[mask])
        drop = len(y[mask]) >= batch_size
        return DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=drop), int(mask.sum())

    def _eval_loss(self, model, arrays, y, mi, batch_size):
        mask = ~np.all(arrays[mi] == MISSING_VALUE, axis=1)
        if mask.sum() == 0:
            return float('inf')
        ds     = _SingleModDataset(arrays[mi][mask], y[mask])
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False)
        model.eval()
        total, n = 0.0, 0
        with torch.no_grad():
            for xb, yb in loader:
                xb, yb = xb.to(self.device), yb.to(self.device)
                pred   = model(xb)
                yb_    = yb.unsqueeze(-1) if yb.dim() == 1 else yb
                total += float(self.criterion(pred, yb_).item()) * yb.size(0)
                n     += yb.size(0)
        return total / n if n > 0 else float('inf')

    def _group_by_availability(self, modalities):
        B    = modalities[0].size(0)
        code = torch.zeros(B, dtype=torch.long, device=modalities[0].device)
        for i, m in enumerate(modalities):
            present = (m != MISSING_VALUE).view(B, -1).all(dim=1)
            code    = code + (present.long() << i)
        patterns = {}
        for c in torch.unique(code).tolist():
            idx     = (code == c).nonzero(as_tuple=True)[0]
            pattern = tuple(bool((c >> i) & 1) for i in range(NUM_MODALITIES))
            patterns[pattern] = idx
        return patterns

    def _weights_for(self, present_models):
        eps    = 1e-8
        losses = [self.best_val_losses[m] for m in present_models]
        finite = [(m, l) for m, l in zip(present_models, losses) if np.isfinite(l)]
        if not finite:
            return None
        inv  = np.array([1.0 / (l + eps) for _, l in finite], dtype=np.float32)
        inv /= inv.sum()
        wmap = {m: float(w) for (m, _), w in zip(finite, inv)}
        w    = np.array([wmap.get(m, 0.0) for m in present_models], dtype=np.float32)
        s    = w.sum()
        if s > 0:
            w /= s
        return torch.tensor(w, dtype=torch.float32, device=self.device)

    def train(self, train_loader, val_loader):
        tr_arrays, y_tr = loader_to_modality_arrays(train_loader)
        va_arrays, y_va = loader_to_modality_arrays(val_loader)
        bs = train_loader.batch_size

        for mi in range(NUM_MODALITIES):
            tr_loader, n_tr = self._avail_loader(tr_arrays, y_tr, mi, bs)
            if n_tr == 0:
                continue

            model      = self.models[mi]
            opt        = self.optimizers[mi]
            best_state = copy.deepcopy(model.state_dict())
            best_loss  = float('inf')

            for _ in range(self.epochs):
                model.train()
                for xb, yb in tr_loader:
                    xb, yb = xb.to(self.device), yb.to(self.device)
                    opt.zero_grad()
                    pred = model(xb)
                    loss = self.criterion(
                        pred, yb.unsqueeze(-1) if yb.dim() == 1 else yb
                    )
                    loss.backward()
                    opt.step()

                val_loss = self._eval_loss(model, va_arrays, y_va, mi, bs)
                if val_loss < best_loss:
                    best_loss  = val_loss
                    best_state = copy.deepcopy(model.state_dict())

            model.load_state_dict(best_state)
            self.best_val_losses[mi] = best_loss

        finite = [(i, l) for i, l in enumerate(self.best_val_losses) if np.isfinite(l)]
        self.ens_idxs = [i for i, _ in sorted(finite, key=lambda x: x[1])]

    def test(self, test_loader, missing_value=None):
        ensemble_methods = ['simple_average', 'weighted_average',
                            'best_single', 'greedy_ensemble']
        records      = {m: {'y_true': [], 'y_pred': []} for m in ensemble_methods}
        cohort_preds = [{'y_true': [], 'y_pred': []} for _ in range(NUM_MODALITIES)]

        for model in self.models:
            model.eval()

        with torch.no_grad():
            for batch in test_loader:
                mods, target = batch[:-1], batch[-1]
                if self.use_gpu:
                    mods   = [m.cuda() for m in mods]
                    target = target.cuda()
                target_f = target.float()
                patterns = self._group_by_availability(mods)
                for pattern, idx in patterns.items():
                    present = [k for k, ok in enumerate(pattern) if ok]
                    if not present:
                        continue
                    tg   = target_f[idx]
                    outs = []
                    for mi in present:
                        out = self.models[mi](mods[mi][idx])
                        outs.append(out)
                        cohort_preds[mi]['y_true'].append(tg.cpu().numpy())
                        cohort_preds[mi]['y_pred'].append(out.cpu().numpy())
                    stack   = torch.stack(outs)
                    weights = self._weights_for(present)
                    for method in ensemble_methods:
                        if method == 'simple_average':
                            final = torch.mean(stack, dim=0)
                        elif method == 'weighted_average':
                            final = (torch.mean(stack, dim=0) if weights is None
                                     else torch.sum(weights.view(-1, 1, 1) * stack, dim=0))
                        elif method == 'best_single':
                            best  = min(present,
                                key=lambda m: self.best_val_losses[m]
                                if np.isfinite(self.best_val_losses[m]) else float('inf'))
                            final = stack[present.index(best)]
                        elif method == 'greedy_ensemble':
                            chosen = [m for m in self.ens_idxs if m in present]
                            if not chosen:
                                final = torch.mean(stack, dim=0)
                            else:
                                cw = self._weights_for(chosen)
                                cp = [present.index(m) for m in chosen]
                                final = (torch.mean(stack[cp], dim=0) if cw is None
                                         else torch.sum(cw.view(-1, 1, 1) * stack[cp], dim=0))
                        else:
                            raise ValueError(method)
                        records[method]['y_true'].append(tg.cpu().numpy())
                        records[method]['y_pred'].append(final.cpu().numpy())

        results = {}
        for method, rec in records.items():
            if not rec['y_true']:
                results[method] = float('nan')
                continue
            yt = np.concatenate(rec['y_true']).ravel()
            yp = np.concatenate(rec['y_pred']).ravel()
            results[method] = float(np.mean((yt - yp) ** 2))
        cohort_mse = []
        for rec in cohort_preds:
            if not rec['y_true']:
                cohort_mse.append(float('nan'))
            else:
                yt = np.concatenate(rec['y_true']).ravel()
                yp = np.concatenate(rec['y_pred']).ravel()
                cohort_mse.append(float(np.mean((yt - yp) ** 2)))
        results['cohort'] = cohort_mse
        return results


# ============================================================
# RESULT FLATTENING & AVERAGING
# ============================================================

def flatten_results(family, training_mode, setting, results, repetition, split_seed):
    rows = []
    for k, v in results.items():
        if k == 'cohort':
            for i, item in enumerate(v):
                rows.append({
                    'repetition':    repetition,
                    'split_seed':    split_seed,
                    'family':        family,
                    'training_mode': training_mode,
                    'setting':       setting,
                    'method':        f'cohort_{i}',
                    'mse':           float(item) if (item is not None
                                     and not (isinstance(item, float) and np.isnan(item)))
                                     else float('nan'),
                })
        else:
            mse_val = v.get('mse', float('nan')) if isinstance(v, dict) else v
            rows.append({
                'repetition':    repetition,
                'split_seed':    split_seed,
                'family':        family,
                'training_mode': training_mode,
                'setting':       setting,
                'method':        k,
                'mse':           float(mse_val) if mse_val is not None else float('nan'),
            })
    return rows


def apply_setting_order(df: pd.DataFrame) -> pd.DataFrame:
    order = [
        'filtered',
        'real_imputation_filteredpart',
        'real_imputation_extrapart',
        'real_imputation_overall',
        'imputation_filteredpart',
        'imputation_extrapart',
        'imputation_overall',
    ]
    df = df.copy()
    df['setting'] = pd.Categorical(df['setting'], categories=order, ordered=True)
    sort_cols = [c for c in ['repetition', 'split_seed', 'setting', 'family',
                              'training_mode', 'method'] if c in df.columns]
    return df.sort_values(sort_cols).reset_index(drop=True)


def _se(series: pd.Series) -> float:
    x = series.dropna().astype(float)
    n = len(x)
    return float('nan') if n <= 1 else float(x.std(ddof=1) / np.sqrt(n))


def average_over_reps(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = ['family', 'training_mode', 'setting', 'method']
    avg = (
        df.groupby(group_cols, dropna=False, observed=True)
        .agg(mse_mean=('mse', 'mean'), mse_se=('mse', _se))
        .reset_index()
    )
    counts = (
        df.groupby(group_cols, dropna=False, observed=True)['repetition']
        .nunique().reset_index(name='num_repetitions')
    )
    avg = avg.merge(counts, on=group_cols, how='left')
    return apply_setting_order(avg)


# ============================================================
# THREE-WAY PARTITION EVALUATION (kept for reactivation)
# ============================================================

def evaluate_three_way(model_obj, family, training_mode, prefix,
                        test_loader, full_obs_mask,
                        all_rows, repetition, split_seed,
                        needs_missing_value=False, missing_value=None):
    bs          = test_loader.batch_size
    extra_mask  = ~full_obs_mask
    filt_loader  = subset_loader_by_mask(test_loader, full_obs_mask, bs)
    extra_loader = subset_loader_by_mask(test_loader, extra_mask,    bs)

    for loader, suffix in [
        (filt_loader,  f'{prefix}_filteredpart'),
        (extra_loader, f'{prefix}_extrapart'),
        (test_loader,  f'{prefix}_overall'),
    ]:
        res = (model_obj.test(loader, missing_value=missing_value)
               if needs_missing_value else model_obj.test(loader))
        all_rows.extend(
            flatten_results(family, training_mode, suffix, res, repetition, split_seed)
        )


# ============================================================
# DATA PREPARATION FOR ONE REPETITION
# ============================================================

def build_all_settings(data_preparer, random_state: int):
    """Filtered-only path. Imputation steps commented out."""
    train_loader, val_loader, test_loader, _, _, _ = \
        data_preparer.get_data_loaders(
            N,
            trans_type=TRANS_TYPE,
            mod_prop=MOD_PROP,
            interactive_prop=INTERACTIVE_PROP,
            dim_modalities=DIM_MODALITIES,
            dim_latent=DIM_LATENT,
            noise_ratios=NOISE_RATIOS,
            random_state=random_state,
        )

    train_miss, val_miss, test_miss = data_preparer.apply_missing_modalities(
        train_loader, val_loader, test_loader,
        modality_fractions=FRACTIONS,
        random_state=0,
        missing_value=MISSING_VALUE,
    )

    train_filt, val_filt, test_filt = data_preparer.filter_fully_observed(
        train_miss, val_miss, test_miss,
        missing_value=MISSING_VALUE,
    )

    # # Required only for the imputation blocks:
    # test_miss_full_obs_mask = get_fully_observed_mask(test_miss)
    # print("Running KNN imputation for this repetition...")
    # train_knn, val_knn, test_knn = knn_impute_no_leakage(
    #     train_miss, val_miss, test_miss
    # )

    return {
        'train_miss': train_miss, 'val_miss': val_miss, 'test_miss': test_miss,
        'train_filt': train_filt, 'val_filt': val_filt, 'test_filt': test_filt,
        # 'train_knn':  train_knn,  'val_knn':  val_knn,  'test_knn':  test_knn,
        # 'test_miss_full_obs_mask': test_miss_full_obs_mask,
    }


# ============================================================
# SINGLE REPETITION
# ============================================================

def _cfg_for(base_config: dict, setting: str) -> dict:
    cfg = copy.deepcopy(base_config)
    cfg['ckpt_dir'] = os.path.join(base_config['ckpt_dir'], setting)
    os.makedirs(cfg['ckpt_dir'], exist_ok=True)
    return cfg


def run_one_repetition(data_preparer, repetition: int, split_seed: int):
    print('\n' + '#' * 80)
    print(f'REPETITION {repetition + 1}/{NUM_REPETITIONS}  |  split_seed={split_seed}')
    print('#' * 80)

    seed_everything(split_seed)
    config, _ = make_config(split_seed)

    data     = build_all_settings(data_preparer, random_state=split_seed)
    all_rows = []

    # ------------------------------------------------------------------ #
    # FILTERED SETTING
    # All active families train AND test on filtered loaders.
    # Active: BenchmarksLateFusion, joint marginal, joint shapley.
    # Disabled: metafusion (rho_search) and metafusion_ablation.
    # ------------------------------------------------------------------ #
    print('\n--- FILTERED ---')
    cfg_f = _cfg_for(config, 'filtered')

    # benchmarks (custom: produces all six late-fusion variants + per-modality + early_fusion)
    bm = BenchmarksLateFusion(cfg_f)
    bm.train(data['train_filt'], data['val_filt'])
    bm_res = bm.test(data['test_filt'])
    all_rows.extend(
        flatten_results('benchmarks', 'na', 'filtered', bm_res, repetition, split_seed)
    )

    # # metafusion — disabled
    # meta_cohort   = build_cohort_new()
    # cohort_models = meta_cohort.get_cohort_models()
    # metafuse = Trainer_new(cfg_f, cohort_models,
    #                        [data['train_filt'], data['val_filt']])
    # metafuse.train()
    # meta_res = metafuse.test(data['test_filt'])
    # all_rows.extend(
    #     flatten_results('metafusion', 'rho_search', 'filtered', meta_res, repetition, split_seed)
    # )
    # metafuse.train_ablation()
    # indep_res = metafuse.test_ablation(data['test_filt'])
    # all_rows.extend(
    #     flatten_results('metafusion_ablation', 'rho0', 'filtered', indep_res, repetition, split_seed)
    # )

    # joint marginal
    joint_cohort  = build_cohort_new()
    cohort_models = joint_cohort.get_cohort_models()
    joint_m = Trainer_Joint_new(cfg_f, cohort_models,
                                [data['train_filt'], data['val_filt']])
    joint_m.train('marginal', missing_value=MISSING_VALUE)
    joint_m_res = joint_m.test(data['test_filt'], missing_value=MISSING_VALUE)
    all_rows.extend(
        flatten_results('joint', 'marginal', 'filtered', joint_m_res, repetition, split_seed)
    )

    # joint shapley
    joint_cohort  = build_cohort_new()
    cohort_models = joint_cohort.get_cohort_models()
    joint_s = Trainer_Joint_new(cfg_f, cohort_models,
                                [data['train_filt'], data['val_filt']])
    joint_s.train('shapley', missing_value=MISSING_VALUE)
    joint_s_res = joint_s.test(data['test_filt'], missing_value=MISSING_VALUE)
    all_rows.extend(
        flatten_results('joint', 'shapley', 'filtered', joint_s_res, repetition, split_seed)
    )

    # ------------------------------------------------------------------ #
    # REAL IMPUTATION SETTING — DISABLED
    # ------------------------------------------------------------------ #
    # print('\n--- REAL IMPUTATION ---')
    # cfg_r = _cfg_for(config, 'real_imputation')
    # mask  = data['test_miss_full_obs_mask']
    #
    # bm_models, bm_dims = build_benchmark_models_and_dims(
    #     data['train_knn'], data['val_knn']
    # )
    # bm = Benchmarks(cfg_r, bm_models,
    #                 [data['train_knn'], data['val_knn']],
    #                 model_dims=bm_dims)
    # bm.train()
    # evaluate_three_way(
    #     bm, 'benchmarks', 'na', 'real_imputation',
    #     data['test_knn'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )
    #
    # meta_cohort   = build_cohort_new()
    # cohort_models = meta_cohort.get_cohort_models()
    # metafuse = Trainer_new(cfg_r, cohort_models,
    #                        [data['train_knn'], data['val_knn']])
    # metafuse.train()
    # evaluate_three_way(
    #     metafuse, 'metafusion', 'rho_search', 'real_imputation',
    #     data['test_knn'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )
    #
    # metafuse.train_ablation()
    # bs = data['test_knn'].batch_size
    # for loader, suffix in [
    #     (subset_loader_by_mask(data['test_knn'], mask,  bs),  'real_imputation_filteredpart'),
    #     (subset_loader_by_mask(data['test_knn'], ~mask, bs),  'real_imputation_extrapart'),
    #     (data['test_knn'],                                     'real_imputation_overall'),
    # ]:
    #     res = metafuse.test_ablation(loader)
    #     all_rows.extend(
    #         flatten_results('metafusion_ablation', 'rho0', suffix, res, repetition, split_seed)
    #     )
    #
    # joint_cohort  = build_cohort_new()
    # cohort_models = joint_cohort.get_cohort_models()
    # joint_m = Trainer_Joint_new(cfg_r, cohort_models,
    #                             [data['train_knn'], data['val_knn']])
    # joint_m.train('marginal', missing_value=None)
    # evaluate_three_way(
    #     joint_m, 'joint', 'marginal', 'real_imputation',
    #     data['test_knn'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )
    #
    # joint_cohort  = build_cohort_new()
    # cohort_models = joint_cohort.get_cohort_models()
    # joint_s = Trainer_Joint_new(cfg_r, cohort_models,
    #                             [data['train_knn'], data['val_knn']])
    # joint_s.train('shapley', missing_value=None)
    # evaluate_three_way(
    #     joint_s, 'joint', 'shapley', 'real_imputation',
    #     data['test_knn'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )

    # ------------------------------------------------------------------ #
    # IMPUTATION (MISSINGNESS-AWARE) SETTING — DISABLED
    # ------------------------------------------------------------------ #
    # print('\n--- IMPUTATION (MISSINGNESS-AWARE) ---')
    # cfg_i = _cfg_for(config, 'imputation')
    #
    # bm_models, bm_dims = build_benchmark_models_and_dims(
    #     data['train_miss'], data['val_miss']
    # )
    # bm = Benchmarks(cfg_i, bm_models,
    #                 [data['train_miss'], data['val_miss']],
    #                 model_dims=bm_dims)
    # bm.train()
    # evaluate_three_way(
    #     bm, 'benchmarks', 'na', 'imputation',
    #     data['test_miss'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )
    #
    # lf = LateFusionAvailableOnly(config)
    # lf.train(data['train_miss'], data['val_miss'])
    # evaluate_three_way(
    #     lf, 'benchmarks_option2', 'late_fusion_available_only', 'imputation',
    #     data['test_miss'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=True, missing_value=MISSING_VALUE,
    # )
    #
    # meta_cohort   = build_cohort_new()
    # cohort_models = meta_cohort.get_cohort_models()
    # metafuse = Trainer_new(cfg_i, cohort_models,
    #                        [data['train_miss'], data['val_miss']])
    # metafuse.train()
    # evaluate_three_way(
    #     metafuse, 'metafusion', 'rho_search', 'imputation',
    #     data['test_miss'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=False,
    # )
    #
    # metafuse.train_ablation()
    # bs = data['test_miss'].batch_size
    # for loader, suffix in [
    #     (subset_loader_by_mask(data['test_miss'], mask,  bs),  'imputation_filteredpart'),
    #     (subset_loader_by_mask(data['test_miss'], ~mask, bs),  'imputation_extrapart'),
    #     (data['test_miss'],                                     'imputation_overall'),
    # ]:
    #     res = metafuse.test_ablation(loader)
    #     all_rows.extend(
    #         flatten_results('metafusion_ablation', 'rho0', suffix, res, repetition, split_seed)
    #     )
    #
    # joint_cohort  = build_cohort_new()
    # cohort_models = joint_cohort.get_cohort_models()
    # joint_m = Trainer_Joint_new(cfg_i, cohort_models,
    #                             [data['train_miss'], data['val_miss']])
    # joint_m.train('marginal', missing_value=MISSING_VALUE)
    # evaluate_three_way(
    #     joint_m, 'joint', 'marginal', 'imputation',
    #     data['test_miss'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=True, missing_value=MISSING_VALUE,
    # )
    #
    # joint_cohort  = build_cohort_new()
    # cohort_models = joint_cohort.get_cohort_models()
    # joint_s = Trainer_Joint_new(cfg_i, cohort_models,
    #                             [data['train_miss'], data['val_miss']])
    # joint_s.train('shapley', missing_value=MISSING_VALUE)
    # evaluate_three_way(
    #     joint_s, 'joint', 'shapley', 'imputation',
    #     data['test_miss'], mask,
    #     all_rows, repetition, split_seed,
    #     needs_missing_value=True, missing_value=MISSING_VALUE,
    # )

    rep_df = pd.DataFrame(all_rows)
    rep_df = apply_setting_order(rep_df)
    print(f'\nFinished repetition {repetition + 1}')
    print(rep_df.head(15))
    return rep_df


# ============================================================
# MAIN
# ============================================================

def run_all():
    data_preparer = PrepareSyntheticData(
        data_name=DATA_NAME, test_size=0.2, val_size=0.2
    )

    all_dfs = []
    for rep_idx, split_seed in enumerate(REPETITION_SEEDS):
        rep_df = run_one_repetition(data_preparer, rep_idx, split_seed)
        all_dfs.append(rep_df)

        interim = pd.concat(all_dfs, ignore_index=True)
        interim = apply_setting_order(interim)
        interim.to_csv(
            os.path.join(OUTDIR, 'simulation_results_raw_interim.csv'), index=False
        )

    raw_df = pd.concat(all_dfs, ignore_index=True)
    raw_df = apply_setting_order(raw_df)
    avg_df = average_over_reps(raw_df)

    raw_path = os.path.join(OUTDIR, 'simulation_results_raw_20reps.csv')
    avg_path = os.path.join(OUTDIR, 'simulation_results_avg_with_se_20reps.csv')

    raw_df.to_csv(raw_path, index=False)
    avg_df.to_csv(avg_path, index=False)

    print('\n' + '=' * 70)
    print('ALL REPETITIONS COMPLETE')
    print(f'Raw results  -> {raw_path}')
    print(f'Avg results  -> {avg_path}')
    print('=' * 70)
    print('\nRAW HEAD:')
    print(raw_df.head(20))
    print('\nAVERAGE HEAD:')
    print(avg_df.head(20))
    return raw_df, avg_df


if __name__ == '__main__':
    seed_everything(SEED)

    print('USE_GPU          =', USE_GPU)
    print('NUM_REPETITIONS  =', NUM_REPETITIONS)
    print('FRACTIONS        =', FRACTIONS)
    print('MISSING_VALUE    =', MISSING_VALUE)

    run_all()

I0000 00:00:1777759533.107942 3005211 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777759533.146235 3005211 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777759536.118448 3005211 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/zmoslemi/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axe

USE_GPU          = False
NUM_REPETITIONS  = 20
FRACTIONS        = [1.0, 0.8, 0.6]
MISSING_VALUE    = 10.0

################################################################################
REPETITION 1/20  |  split_seed=1234
################################################################################


/home/zmoslemi/parnian/meta_fusion/utils.py:36: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.data = torch.tensor(data, dtype=torch.float32)



--- FILTERED ---
Training BenchmarksLateFusion (regression)
  Training unimodal model 1/3 (input dim = 200)
    best val MSE = 360.3284
  Training unimodal model 2/3 (input dim = 300)
    best val MSE = 318.3265
  Training unimodal model 3/3 (input dim = 100)
    best val MSE = 331.3516
  Training early-fusion model (input dim = 600)
    best val MSE = 260.6913
  Greedy ensemble subset (val-MSE selected): [1, 2]
  Method: (modality_1), Test_MSE: 396.2477
  Method: (modality_2), Test_MSE: 377.0754
  Method: (modality_3), Test_MSE: 373.2579
  Method: (early_fusion), Test_MSE: 259.9982
  Method: (late_fusion_simple_average), Test_MSE: 333.8564
  Method: (late_fusion_weighted_average), Test_MSE: 332.6171
  Method: (late_fusion_best_single), Test_MSE: 377.0754
  Method: (late_fusion_greedy_ensemble), Test_MSE: 323.0476
  Method: (late_fusion_majority_voting), Test_MSE: 360.3710
  Method: (late_fusion_weighted_voting), Test_MSE: 360.3710
Start training student cohort...

Epoch: 1/10 - LR: 0

100%|█████| 619/619 [00:00<00:00, 3399.18it/s, loss=489.2213, batch_time=0.018s]


...


Epoch: 9/10 - LR: 0.001000
Using exact Shapley computation


100%|█| 619/619 [00:00<00:00, 1941.09it/s, avg_loss=282.4668, batch_time=0.032s]



Epoch: 10/10 - LR: 0.001000
Using exact Shapley computation


100%|█| 619/619 [00:00<00:00, 2041.12it/s, avg_loss=267.8406, batch_time=0.030s]


Finished training student cohort!
Selecting greedy ensemble on the best cohort...
Pruned 1 worst models, keeping 2 models
Initial best models: [1, 2] with losses: [tensor(369.4897, grad_fn=<MseLossBackward0>), tensor(444.2350, grad_fn=<MseLossBackward0>)]
Done!
Method: (simple_average), Test_MSE: 287.7113342285156
Method: (weighted_average), Test_MSE: 282.9217529296875
Method: (greedy_ensemble), Test_MSE: 282.5285949707031
Method: (best_single), Test_MSE: 350.94189453125
Method: (cohort), Test_MSE: [405.43536376953125, 350.94189453125, 403.998046875]

Finished repetition 20
    repetition  split_seed      family training_mode   setting  \
0           19        1253  benchmarks            na  filtered   
1           19        1253  benchmarks            na  filtered   
2           19        1253  benchmarks            na  filtered   
3           19        1253  benchmarks            na  filtered   
4           19        1253  benchmarks            na  filtered   
5           19        1